# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install --upgrade pip --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import platform
import os
import subprocess

### 1. Jemula Setup
#### 1.1. Class Paths

In [ ]:
# --- Jemula Class Paths ---
# jemula and jemula802 are one repository, cloned into PROJECT_DIR below.
# Every path here is built from it, so moving the checkout means editing one
# line.
PROJECT_DIR = "jemula-project"
REPO_URL = "https://gitlab.ethz.ch/wireless/jemula.git"

lib = f"{PROJECT_DIR}/simulator/kernel/lib"
# webui holds the page every run writes next to its results (timeline.html).
common = f"./:{PROJECT_DIR}/simulator/engine/bin:{PROJECT_DIR}/simulator/kernel/bin:{PROJECT_DIR}/simulator/engine/webui"
classpath_jemula = f"{common}:{lib}/xstream-1.3.1.jar"
classpath_jemula802 = f"{classpath_jemula}:{lib}/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")

#### 1.2. Cloning the Repository

In [ ]:
# --- Clone or Update the Jemula Repository ---
# One repository holding both jemula and jemula802, with the tests, the
# scenarios and the fixes.
if os.path.isdir(f"{PROJECT_DIR}/.git"):
    print(f"Updating {PROJECT_DIR} ...")
    if subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"]).returncode == 0:
        print(f"{PROJECT_DIR} updated.")
    else:
        # The update does not fit on top of this copy - usually because it holds
        # files you changed (a scenario's seed, an algorithm) that the update
        # changes too. Keep that copy, with your changes, and start afresh.
        import time
        kept = f"{PROJECT_DIR}-{time.strftime('%Y%m%d-%H%M%S')}"
        os.rename(PROJECT_DIR, kept)
        print(f"Could not update {PROJECT_DIR}. Your copy, with your changes, is kept as {kept}.")
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
        print(f"{PROJECT_DIR} cloned.")
else:
    print(f"Cloning {PROJECT_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    print(f"{PROJECT_DIR} cloned.")


#### 1.3. Compiling Jemula

This compiles the whole simulator once. Your own algorithms in `my-simulations/algorithms/` are compiled again by `Jemula802_Run.ipynb` before every run.

In [ ]:
# --- Compile Jemula and Jemula802 ---
ROOT_DIR = os.getcwd()
print(f"Root directory: {ROOT_DIR}")

compile_jobs = [
    {
        "classpath": classpath_jemula,
        "out": f"{PROJECT_DIR}/simulator/kernel/bin",
        "src": [
            f"{PROJECT_DIR}/simulator/kernel/src/plot/*.java",
            f"{PROJECT_DIR}/simulator/kernel/src/kernel/*.java",
            f"{PROJECT_DIR}/simulator/kernel/src/statistics/*.java",
        ],
        "name": "jemula",
    },
    {
        "classpath": classpath_jemula802,
        "out": f"{PROJECT_DIR}/simulator/engine/bin",
        "src": [
            f"{PROJECT_DIR}/simulator/engine/src/emulator/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer0_medium/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer1_802Phy/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer1_80211Phy/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer2_802Mac/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer2_80211Mac/*.java",
            f"{PROJECT_DIR}/my-simulations/algorithms/layer2_802Algorithms/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer3_network/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer4_transport/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/layer5_application/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/station/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/visualization/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/util/*.java",
            f"{PROJECT_DIR}/simulator/engine/src/gui/*.java",
        ],
        "name": "jemula802",
    },
]

for job in compile_jobs:
    print(f"Compiling {job['name']} ...")
    cmd = ["javac", "-g:none", "-classpath", job["classpath"], "-d", job["out"]] + job["src"]
    result = subprocess.run(" ".join(cmd), shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout + result.stderr)
        raise SystemExit(f"Compiling {job['name']} failed - see the messages above.")
    print(f"Compilation of {job['name']} done.\n")